Лабораторна робота №4
Тема: Візуалізація даних 2
Виконав: Сиротін Максим ФБ-42

Мета: Створити інтерактивну програму для малювання графіка функції гармоніки з накладеним шумом, використовуючи слайдери, чекбокси та кнопки. Реалізувати фільтрацію зашумленого сигналу.

In [1]:
%pip install matplotlib numpy scipy -q

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.widgets import Slider, Button, CheckButtons
from scipy.signal import iirfilter, filtfilt

%matplotlib tk


[notice] A new release of pip is available: 25.0.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.


In [2]:
%matplotlib tk
initial_amplitude = 1.0
initial_frequency = 1.0
initial_phase = 0.0
initial_noise_mean = 0.0
initial_noise_cov = 0.1
initial_cutoff = 5.0

t = np.linspace(0, 10, 1000)

current_noise = np.zeros_like(t)
last_mean = None
last_cov = None

def get_noise(mean, cov):
    global current_noise, last_mean, last_cov
    if mean != last_mean or cov != last_cov:
        current_noise = np.random.normal(mean, np.sqrt(max(cov, 0.001)), len(t))
        last_mean = mean
        last_cov = cov
    return current_noise

def harmonic_with_noise(t, amplitude, frequency, phase, noise_mean, noise_covariance, show_noise):
    y_pure = amplitude * np.sin(frequency * t + phase)
    if show_noise:
        noise = get_noise(noise_mean, noise_covariance)
        y_noisy = y_pure + noise
    else:
        y_noisy = y_pure
    return y_noisy, y_pure

def filter_signal(data, cutoff, fs=100.0):
    nyq = 0.5 * fs
    normal_cutoff = cutoff / nyq
    normal_cutoff = np.clip(normal_cutoff, 0.01, 0.99)
    b, a = iirfilter(4, normal_cutoff, btype='low', ftype='butter')
    return filtfilt(b, a, data)

fig, ax = plt.subplots(figsize=(10, 6))
plt.subplots_adjust(left=0.1, bottom=0.4) 

y_noisy, y_pure = harmonic_with_noise(t, initial_amplitude, initial_frequency, initial_phase, 
                                      initial_noise_mean, initial_noise_cov, True)
y_filtered = filter_signal(y_noisy, initial_cutoff)

line_noisy, = ax.plot(t, y_noisy, label='Зашумлений сигнал', color='orange', alpha=0.7)
line_pure, = ax.plot(t, y_pure, label='Чиста гармоніка', color='black', linestyle='--')
line_filtered, = ax.plot(t, y_filtered, label='Відфільтрований сигнал', color='blue', linewidth=2)

ax.set_title('Гармоніка з шумом та фільтрацією')
ax.set_xlabel('Час (t)')
ax.set_ylabel('y(t)')
ax.legend(loc='upper right')
ax.grid(True)

axcolor = 'lightgoldenrodyellow'

ax_amp = plt.axes([0.15, 0.3, 0.65, 0.03], facecolor=axcolor)
ax_freq = plt.axes([0.15, 0.25, 0.65, 0.03], facecolor=axcolor)
ax_phase = plt.axes([0.15, 0.2, 0.65, 0.03], facecolor=axcolor)
ax_mean = plt.axes([0.15, 0.15, 0.65, 0.03], facecolor=axcolor)
ax_cov = plt.axes([0.15, 0.1, 0.65, 0.03], facecolor=axcolor)
ax_cutoff = plt.axes([0.15, 0.05, 0.65, 0.03], facecolor=axcolor)

s_amp = Slider(ax_amp, 'Amplitude', 0.1, 5.0, valinit=initial_amplitude)
s_freq = Slider(ax_freq, 'Frequency', 0.1, 10.0, valinit=initial_frequency)
s_phase = Slider(ax_phase, 'Phase', 0.0, 2*np.pi, valinit=initial_phase)
s_mean = Slider(ax_mean, 'Noise Mean', -2.0, 2.0, valinit=initial_noise_mean)
s_cov = Slider(ax_cov, 'Noise Covariance', 0.0, 2.0, valinit=initial_noise_cov)
s_cutoff = Slider(ax_cutoff, 'Cutoff Freq', 0.1, 20.0, valinit=initial_cutoff)

ax_check = plt.axes([0.85, 0.15, 0.1, 0.1])
check = CheckButtons(ax_check, ['Show Noise'], [True])

ax_reset = plt.axes([0.85, 0.05, 0.1, 0.04])
btn_reset = Button(ax_reset, 'Reset', color=axcolor, hovercolor='0.975')

def update(val):
    amp = s_amp.val
    freq = s_freq.val
    phase = s_phase.val
    mean = s_mean.val
    cov = s_cov.val
    cutoff = s_cutoff.val
    show_noise = check.get_status()[0]
    yn, yp = harmonic_with_noise(t, amp, freq, phase, mean, cov, show_noise)
    yf = filter_signal(yn, cutoff)
    line_noisy.set_ydata(yn)
    line_pure.set_ydata(yp)
    line_filtered.set_ydata(yf)
    ax.relim()
    ax.autoscale_view()
    fig.canvas.draw_idle()

s_amp.on_changed(update)
s_freq.on_changed(update)
s_phase.on_changed(update)
s_mean.on_changed(update)
s_cov.on_changed(update)
s_cutoff.on_changed(update)
check.on_clicked(update)

def reset(event):
    s_amp.reset()
    s_freq.reset()
    s_phase.reset()
    s_mean.reset()
    s_cov.reset()
    s_cutoff.reset()

    if not check.get_status()[0]:
        check.set_active(0)
btn_reset.on_clicked(reset)

plt.show()

Інструкція користувача
1. Керування гармонікою: Використовуйте повзунки `Amplitude`, `Frequency` та `Phase` для зміни форми початкового (чистого) сигналу.
2. Керування шумом: Повзунки `Noise Mean` (середнє значення) та `Noise Covariance` (дисперсія) дозволяють змінювати характер накладеного шуму. Шум генерується наново лише при зміні цих параметрів.
3. Фільтрація: Повзунок `Cutoff Freq` відповідає за частоту зрізу IIR-фільтра. Зменшуйте це значення, щоб сильніше згладити зашумлений сигнал.
4. Перемикач шуму: Чекбокс `Show Noise` дозволяє миттєво увімкнути або вимкнути відображення зашумленого сигналу.
5. Скидання: Натисніть кнопку `Reset`, щоб повернути всі параметри до початкових значень.